Problem Statement :
Hospital Patient Data Analysis
Context:
A hospital maintains patient records including admission details, department, diagnosis, doctor, and bill amount. You have two datasets: one with patient info and another with billing details. Some patients have blank bill amounts, and there are multiple rows for the same patient due to follow-ups.
Tasks:
    1. Load the patient dataset and show summary with info().
    2. Select only the columns relevant for billing: ['PatientID', 'Department', 'Doctor', 'BillAmount'].
    3. Drop administrative columns like ['ReceptionistID', 'CheckInTime'].
    4. Use groupby to find total bill amount per department.
    5. Remove duplicate patient records based on PatientID.
    6. Fill missing BillAmount values with the mean bill amount.
    7. Merge the billing dataset with patient dataset on PatientID.
    8. Concatenate an additional DataFrame that contains new patients for the current week (row-wise).
    9. Concatenate new billing category columns like ['InsuranceCovered', 'FinalAmount'] (column-wise).
Expected Outcome:
    • Final cleaned dataset with accurate billing info.
    • All missing values handled, merged dataset across PatientID.
    • Ability to perform further analytics on department-wise revenue or doctor performance.


In [1]:
import pandas as pd

# Load the patient dataset
patient_df = pd.read_csv('/content/Patient_Data.csv')

# Show summary with info()
patient_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


In [2]:
# Select only the columns relevant for billing
billing_relevant_df = patient_df[['PatientID', 'Department', 'Doctor', 'BillAmount']]

# Drop administrative columns
patient_df = patient_df.drop(columns=['ReceptionistID', 'CheckInTime'])

print("DataFrame after selecting billing-relevant columns:")
print(billing_relevant_df.head())
print("\nDataFrame after dropping administrative columns:")
print(patient_df.head())

DataFrame after selecting billing-relevant columns:
   PatientID   Department     Doctor  BillAmount
0        101   Cardiology  Dr. Smith      5000.0
1        102    Neurology   Dr. John         NaN
2        103  Orthopedics    Dr. Lee      7500.0
3        104   Cardiology  Dr. Smith      6200.0
4        105  Dermatology   Dr. Rose         NaN

DataFrame after dropping administrative columns:
   PatientID     Name   Department     Doctor  BillAmount
0        101    Alice   Cardiology  Dr. Smith      5000.0
1        102      Bob    Neurology   Dr. John         NaN
2        103  Charlie  Orthopedics    Dr. Lee      7500.0
3        104    David   Cardiology  Dr. Smith      6200.0
4        105      Eva  Dermatology   Dr. Rose         NaN


In [3]:
# Use groupby to find total bill amount per department
total_bill_per_department = patient_df.groupby('Department')['BillAmount'].sum().reset_index()
print("Total bill amount per department:")
display(total_bill_per_department)

Total bill amount per department:


,Department,BillAmount
0,Cardiology,16200.0
1,Dermatology,0.0
2,Neurology,0.0
3,Orthopedics,7500.0


In [4]:
# Remove duplicate patient records based on PatientID
patient_df_no_duplicates = patient_df.drop_duplicates(subset=['PatientID'])
print("DataFrame after removing duplicate patient records:")
display(patient_df_no_duplicates)

DataFrame after removing duplicate patient records:


,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN


In [5]:
# Fill missing BillAmount values with the mean bill amount
mean_bill_amount = patient_df_no_duplicates['BillAmount'].mean()
patient_df_cleaned = patient_df_no_duplicates.copy()
patient_df_cleaned['BillAmount'].fillna(mean_bill_amount, inplace=True)

print("DataFrame after filling missing BillAmount values:")
display(patient_df_cleaned)

print("Check for any remaining missing BillAmount values:")
print(patient_df_cleaned['BillAmount'].isnull().sum())

DataFrame after filling missing BillAmount values:


/tmp/ipykernel_29329/591942808.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  patient_df_cleaned['BillAmount'].fillna(mean_bill_amount, inplace=True)


,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000
1,102,Bob,Neurology,Dr. John,6233.333333
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000
3,104,David,Cardiology,Dr. Smith,6200.000000
4,105,Eva,Dermatology,Dr. Rose,6233.333333


Check for any remaining missing BillAmount values:
0


In [8]:
# Load the billing dataset
billing_df = pd.read_csv('/content/Billing_Data.csv')

# Merge the billing dataset with the patient dataset on PatientID
merged_df = pd.merge(patient_df_cleaned, billing_df, on='PatientID', how='left')

print("DataFrame after merging billing and patient data:")
display(merged_df)
print("Check for any remaining missing values in the merged DataFrame:")
print(merged_df.isnull().sum())

DataFrame after merging billing and patient data:


,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000,2000,3000
1,102,Bob,Neurology,Dr. John,6233.333333,1500,3500
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000,2500,5000
3,104,David,Cardiology,Dr. Smith,6200.000000,3000,3200
4,105,Eva,Dermatology,Dr. Rose,6233.333333,1000,4000


Check for any remaining missing values in the merged DataFrame:
PatientID           0
Name                0
Department          0
Doctor              0
BillAmount          0
InsuranceCovered    0
FinalAmount         0
dtype: int64


In [9]:
# Create a DataFrame for new patients for the current week
new_patients_df = pd.DataFrame({
    'PatientID': [106, 107],
    'Name': ['Frank', 'Grace'],
    'Department': ['Pediatrics', 'Oncology'],
    'Doctor': ['Dr. White', 'Dr. Green'],
    'BillAmount': [4500.0, 9000.0],
    'InsuranceCovered': [1500, 3000],
    'FinalAmount': [3000, 6000]
})

# Concatenate the new patients DataFrame (row-wise)
final_df_row_wise = pd.concat([merged_df, new_patients_df], ignore_index=True)

print("DataFrame after concatenating new patients (row-wise):")
display(final_df_row_wise)

DataFrame after concatenating new patients (row-wise):


,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000,2000,3000
1,102,Bob,Neurology,Dr. John,6233.333333,1500,3500
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000,2500,5000
3,104,David,Cardiology,Dr. Smith,6200.000000,3000,3200
4,105,Eva,Dermatology,Dr. Rose,6233.333333,1000,4000
5,106,Frank,Pediatrics,Dr. White,4500.000000,1500,3000
6,107,Grace,Oncology,Dr. Green,9000.000000,3000,6000


In [10]:
# Create new billing category columns (e.g., Discount, TaxAmount) as a separate DataFrame
# Ensure the index matches final_df_row_wise for correct column-wise concatenation
additional_billing_columns = pd.DataFrame({
    'Discount': [100, 50, 75, 120, 80, 90, 60],
    'TaxAmount': [200, 150, 220, 180, 250, 170, 210]
}, index=final_df_row_wise.index)

# Concatenate new billing category columns (column-wise)
final_cleaned_df = pd.concat([final_df_row_wise, additional_billing_columns], axis=1)

print("Final cleaned dataset with accurate billing info (after column-wise concatenation):")
display(final_cleaned_df)

print("Check for any remaining missing values in the final DataFrame:")
print(final_cleaned_df.isnull().sum())

Final cleaned dataset with accurate billing info (after column-wise concatenation):


,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount,Discount,TaxAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000,2000,3000,100,200
1,102,Bob,Neurology,Dr. John,6233.333333,1500,3500,50,150
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000,2500,5000,75,220
3,104,David,Cardiology,Dr. Smith,6200.000000,3000,3200,120,180
4,105,Eva,Dermatology,Dr. Rose,6233.333333,1000,4000,80,250
5,106,Frank,Pediatrics,Dr. White,4500.000000,1500,3000,90,170
6,107,Grace,Oncology,Dr. Green,9000.000000,3000,6000,60,210


Check for any remaining missing values in the final DataFrame:
PatientID           0
Name                0
Department          0
Doctor              0
BillAmount          0
InsuranceCovered    0
FinalAmount         0
Discount            0
TaxAmount           0
dtype: int64
